In [1]:
from dotenv import load_dotenv
from pathlib import Path
import os

# Load .env from parent folder
load_dotenv("../.env")

print("OPENAI:", os.getenv("OPENAI_API_KEY")[:10])

OPENAI: sk-proj-AE


In [13]:
import math
import os
import time
from typing import Any, Iterable

from openai import OpenAI
from supabase import create_client


# =========================
# CONFIG
# =========================
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_SERVICE_ROLE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]

TABLE_NAME = "kb_parts"
SCHEMA_NAME = "public"   # change if needed

ID_COLUMN = "id"
TITLE_COLUMN = "title"
KEYWORDS_COLUMN = "keywords"
EMBED_COLUMN = "embedding"

EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM = 1536

READ_BATCH_SIZE = 1000
EMBED_BATCH_SIZE = 50
UPDATE_BATCH_SIZE = 200


# =========================
# HELPERS
# =========================
def clean_text(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()


def build_embedding_text(row: dict[str, Any]) -> str:
    """
    Embed title + keywords.
    """
    title = clean_text(row.get(TITLE_COLUMN))
    keywords = clean_text(row.get(KEYWORDS_COLUMN))

    parts = []
    if title:
        parts.append(f"Title: {title}")
    if keywords:
        parts.append(f"Keywords: {keywords}")

    return "\n".join(parts)


def chunked(seq: list[Any], size: int) -> Iterable[list[Any]]:
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def get_embeddings(client: OpenAI, texts: list[str]) -> list[list[float]]:
    resp = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts,
    )
    vectors = [item.embedding for item in resp.data]

    for idx, vec in enumerate(vectors):
        if len(vec) != EMBED_DIM:
            raise ValueError(
                f"Embedding dimension mismatch at item {idx}: "
                f"expected {EMBED_DIM}, got {len(vec)}"
            )

    return vectors


def fetch_all_rows(supabase_client) -> list[dict[str, Any]]:
    """
    Read all rows from Supabase in pages.
    """
    all_rows: list[dict[str, Any]] = []
    start = 0

    while True:
        end = start + READ_BATCH_SIZE - 1

        query = (
            supabase_client
            .table(TABLE_NAME)
            .select(f"{ID_COLUMN}, {TITLE_COLUMN}, {KEYWORDS_COLUMN}")
            .range(start, end)
        )

        result = query.execute()
        rows = result.data or []

        if not rows:
            break

        all_rows.extend(rows)

        print(f"Fetched {len(all_rows)} rows so far...")
        if len(rows) < READ_BATCH_SIZE:
            break

        start += READ_BATCH_SIZE

    return all_rows


def update_rows_one_by_one(supabase_client, records: list[dict[str, Any]]) -> None:
    """
    Update embedding back into the same table using row id.
    Safer than bulk upsert when you only want to touch one column.
    """
    total = len(records)

    for batch_no, batch in enumerate(chunked(records, UPDATE_BATCH_SIZE), start=1):
        print(
            f"Updating batch {batch_no} / {math.ceil(total / UPDATE_BATCH_SIZE)} "
            f"({len(batch)} rows)"
        )

        for rec in batch:
            (
                supabase_client
                .table(TABLE_NAME)
                .update({EMBED_COLUMN: rec[EMBED_COLUMN]})
                .eq(ID_COLUMN, rec[ID_COLUMN])
                .execute()
            )


def main() -> None:
    # -------------------------
    # Init clients
    # -------------------------
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

    # If your table is NOT in public schema, uncomment this:
    # supabase.postgrest.schema(SCHEMA_NAME)

    # -------------------------
    # Read existing rows
    # -------------------------
    rows = fetch_all_rows(supabase)

    if not rows:
        print("No rows found.")
        return

    print(f"Total rows fetched: {len(rows)}")

    # -------------------------
    # Build embedding text from title + keywords
    # -------------------------
    texts = [build_embedding_text(row) for row in rows]

    print("Sample embedding text:")
    print(texts[0][:500] if texts else "(no text)")
    print("-" * 60)

    # -------------------------
    # Generate embeddings
    # -------------------------
    all_vectors: list[list[float]] = []

    for batch_no, batch in enumerate(chunked(texts, EMBED_BATCH_SIZE), start=1):
        print(
            f"Embedding batch {batch_no} / {math.ceil(len(texts) / EMBED_BATCH_SIZE)} "
            f"({len(batch)} rows)"
        )
        batch_vectors = get_embeddings(openai_client, batch)
        all_vectors.extend(batch_vectors)
        time.sleep(0.15)

    if len(all_vectors) != len(rows):
        raise RuntimeError("Embedding count does not match fetched row count.")

    # -------------------------
    # Prepare update records
    # -------------------------
    update_records: list[dict[str, Any]] = []
    for i, row in enumerate(rows):
        update_records.append({
            ID_COLUMN: row[ID_COLUMN],
            EMBED_COLUMN: all_vectors[i],
        })

    # -------------------------
    # Update embeddings in Supabase
    # -------------------------
    update_rows_one_by_one(supabase, update_records)

    print(f"Done. Updated {len(update_records)} rows in {SCHEMA_NAME}.{TABLE_NAME}")


if __name__ == "__main__":
    main()

Fetched 69 rows so far...
Total rows fetched: 69
Sample embedding text:
Title: ชุดซีลหัวฉีดไทรทัน มีอะไหล่อะไรบ้าง?
Keywords: ชุดซีลหัวฉีด, triton, ไทรทัน, ยางปลอก, ประเก็น, 4D56
------------------------------------------------------------
Embedding batch 1 / 2 (50 rows)
Embedding batch 2 / 2 (19 rows)
Updating batch 1 / 1 (69 rows)
Done. Updated 69 rows in public.kb_parts
